# Advanced Topics and Integration Patterns

This notebook demonstrates the current advanced runtime layer without relying on removed plotting helpers or outdated API paths.

## What This Notebook Covers

- selecting `CompensationStrategy` presets
- analyzing the same workload under different compensators
- using NumPy bridge helpers from `balansis.numpy_integration`
- interpreting explicit compensation signals

In [ ]:
import numpy as np

from balansis import AbsoluteValue, Operations
from balansis.logic.compensator import Compensator, CompensationStrategy
from balansis.numpy_integration import (
    compensated_array_add,
    compensated_dot_product,
    compensated_softmax,
)

print('Advanced runtime and integration imports loaded.')

## Strategy Presets

The current `CompensationStrategy` API exposes preset constructors instead of the older custom boolean-heavy configuration style.

In [ ]:
high_precision = CompensationStrategy.high_precision()
balanced = CompensationStrategy.balanced()
fast = CompensationStrategy.fast()

print('high_precision =', high_precision)
print('balanced =', balanced)
print('fast =', fast)

## Compare Strategies on the Same Input

In [ ]:
values = [
    AbsoluteValue.from_float(1e-10),
    AbsoluteValue.from_float(-1e10),
    AbsoluteValue.from_float(1.0),
    AbsoluteValue.from_float(-1e-15),
]

for name, strategy in [
    ('high_precision', high_precision),
    ('balanced', balanced),
    ('fast', fast),
]:
    compensator = Compensator(strategy=strategy)
    print(name, 'stability =', compensator.analyze_stability(values))

## Low-Level vs Higher-Level API

The tuple-returning `Operations` layer remains the canonical low-level runtime entrypoint.

In [ ]:
ops_result, ops_compensation = Operations.sequence_sum(values)
compensator = Compensator(strategy=balanced)
corrected = compensator.compensate_addition(
    AbsoluteValue.from_float(10.0),
    AbsoluteValue.from_float(-9.999999999999),
)

print('Operations.sequence_sum(values) =', ops_result, ops_compensation)
print('Compensator.compensate_addition(...) =', corrected)

## NumPy Bridge Helpers

The integration layer exposes helper functions rather than a custom NumPy dtype ecosystem.

In [ ]:
left = np.array([1e16, 1.0, -1e16], dtype=np.float64)
right = np.array([0.5, -0.5, 2.0], dtype=np.float64)
logits = np.array([1000.0, 999.0, 998.0], dtype=np.float64)

print('compensated_array_add(left, right) =', compensated_array_add(left, right))
print('compensated_dot_product(left, right) =', compensated_dot_product(left, right))
print('compensated_softmax(logits) =', compensated_softmax(logits))

## Reading Path

Use `docs/guides/integration-patterns.md` and `docs/api/integrations/numpy.md` for the canonical documentation behind these runtime helpers.